# DuckDuckGo Search Tool - Testing & Debugging Notebook

This notebook is designed to test and debug the DuckDuckGo search tool implementation.
We'll explore the API responses, test various queries, and identify any issues with result parsing.

## Setup & Imports

In [2]:
!pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (45 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (250 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]4/5 [requests]


In [3]:
import requests
import json
import logging
from typing import Optional, Dict, Union
from pprint import pprint

# Set up logging to see detailed information
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## 1. Raw API Test - Understanding DuckDuckGo Response

In [4]:
def test_raw_api(query: str):
    """Test raw API response without processing"""
    url = "https://api.duckduckgo.com/"
    params = {
        "q": query,
        "format": "json",
        "no_html": 1,
        "skip_disambig": 1,
    }
    
    print(f"\n{'='*60}")
    print(f"Testing Query: '{query}'")
    print(f"URL: {url}")
    print(f"Params: {params}")
    print(f"{'='*60}\n")
    
    try:
        response = requests.get(url, params=params, timeout=10)
        print(f"Status Code: {response.status_code}")
        print(f"Headers: {dict(response.headers)}\n")
        
        data = response.json()
        print(f"\n📋 TOP-LEVEL KEYS:")
        for key in data.keys():
            value_type = type(data[key]).__name__
            if isinstance(data[key], (str, int, bool)):
                print(f"  - {key}: {value_type} = {repr(data[key])[:100]}")
            elif isinstance(data[key], list):
                print(f"  - {key}: {value_type} (length: {len(data[key])})")
            elif isinstance(data[key], dict):
                print(f"  - {key}: {value_type} (keys: {list(data[key].keys())})")
            else:
                print(f"  - {key}: {value_type}")
        
        return data
    
    except requests.exceptions.RequestException as e:
        print(f"❌ Request Error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ JSON Parse Error: {e}")
        return None

# Test with a simple query
response_data = test_raw_api("python programming")

2026-09-20 14:26:42,118 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443



Testing Query: 'python programming'
URL: https://api.duckduckgo.com/
Params: {'q': 'python programming', 'format': 'json', 'no_html': 1, 'skip_disambig': 1}



2026-09-20 14:26:42,320 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=python+programming&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None


Status Code: 202
Headers: {'Server': 'nginx', 'Date': 'Sun, 20 Sep 2026 14:26:42 GMT', 'Content-Type': 'application/x-javascript', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'Server-Timing': 'total;dur=51;desc="Backend Total [n]"', 'X-Robots-Tag': 'noindex', 'Strict-Transport-Security': 'max-age=31536000', 'Permissions-Policy': 'interest-cohort=()', 'Content-Security-Policy': "default-src 'none' ; connect-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; manifest-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; media-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzcz

## 2. Explore Response Structure

In [5]:
if response_data:
    print("\n📄 ABSTRACT SECTION:")
    print(f"  AbstractText: {response_data.get('AbstractText', 'N/A')[:150]}...")
    print(f"  AbstractSource: {response_data.get('AbstractSource', 'N/A')}")
    print(f"  AbstractURL: {response_data.get('AbstractURL', 'N/A')}")
    
    print("\n📌 RELATED TOPICS:")
    related_topics = response_data.get('RelatedTopics', [])
    print(f"  Total Topics: {len(related_topics)}")
    
    if related_topics:
        print(f"\n  First 3 Topics:")
        for idx, topic in enumerate(related_topics[:3]):
            print(f"\n    [{idx}] Keys: {list(topic.keys())}")
            print(f"        Text: {topic.get('Text', 'N/A')}")
            print(f"        FirstURL: {topic.get('FirstURL', 'N/A')}")
            if 'Topics' in topic:
                print(f"        Has nested Topics: {len(topic['Topics'])} items")
    
    print("\n📊 RESULTS ARRAY:")
    results = response_data.get('Results', [])
    print(f"  Total Results: {len(results)}")
    if results:
        for idx, result in enumerate(results[:3]):
            print(f"\n  [{idx}] {result}")


📄 ABSTRACT SECTION:
  AbstractText: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of signifi...
  AbstractSource: Wikipedia
  AbstractURL: https://en.wikipedia.org/wiki/Python_(programming_language)

📌 RELATED TOPICS:
  Total Topics: 21

  First 3 Topics:

    [0] Keys: ['FirstURL', 'Icon', 'Result', 'Text']
        Text: Python (programming language) Category
        FirstURL: https://duckduckgo.com/c/Python_(programming_language)

    [1] Keys: ['FirstURL', 'Icon', 'Result', 'Text']
        Text: Pydoc - Pydoc is the standard documentation module for the programming language Python.
        FirstURL: https://duckduckgo.com/Pydoc

    [2] Keys: ['FirstURL', 'Icon', 'Result', 'Text']
        Text: NumPy - NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to

## 3. Test Multiple Query Types

In [6]:
# Test different types of queries to see how API responds
test_queries = [
    "python programming",  # Generic topic
    "what is machine learning",  # Question format
    "Python 3.11 release date",  # Specific info
    "zxcvbnmasdfghjkl",  # Nonsense query
]

results_comparison = {}

for query in test_queries:
    data = test_raw_api(query)
    if data:
        results_comparison[query] = {
            'has_abstract': bool(data.get('AbstractText')),
            'abstract_length': len(data.get('AbstractText', '')),
            'related_topics_count': len(data.get('RelatedTopics', [])),
            'results_count': len(data.get('Results', [])),
        }

print("\n\n📊 QUERY COMPARISON:")
print("Query | Abstract | Abstract Len | Topics | Results")
print("-" * 70)
for query, stats in results_comparison.items():
    print(f"{query:30} | {str(stats['has_abstract']):8} | {stats['abstract_length']:12} | {stats['related_topics_count']:6} | {stats['results_count']}")

2026-09-20 14:28:35,185 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443
2026-09-20 14:28:35,380 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=python+programming&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None
2026-09-20 14:28:35,385 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443



Testing Query: 'python programming'
URL: https://api.duckduckgo.com/
Params: {'q': 'python programming', 'format': 'json', 'no_html': 1, 'skip_disambig': 1}

Status Code: 202
Headers: {'Server': 'nginx', 'Date': 'Sun, 20 Sep 2026 14:28:35 GMT', 'Content-Type': 'application/x-javascript', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'Server-Timing': 'total;dur=40;desc="Backend Total [n]"', 'X-Robots-Tag': 'noindex', 'Strict-Transport-Security': 'max-age=31536000', 'Permissions-Policy': 'interest-cohort=()', 'Content-Security-Policy': "default-src 'none' ; connect-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; manifest-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.co

2026-09-20 14:28:35,537 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=what+is+machine+learning&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None
2026-09-20 14:28:35,541 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443
2026-09-20 14:28:35,734 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=Python+3.11+release+date&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None
2026-09-20 14:28:35,738 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443


Status Code: 202
Headers: {'Server': 'nginx', 'Date': 'Sun, 20 Sep 2026 14:28:35 GMT', 'Content-Type': 'application/x-javascript', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'Server-Timing': 'total;dur=39;desc="Backend Total [n]"', 'X-Robots-Tag': 'noindex', 'Strict-Transport-Security': 'max-age=31536000', 'Permissions-Policy': 'interest-cohort=()', 'Content-Security-Policy': "default-src 'none' ; connect-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; manifest-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; media-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzcz

2026-09-20 14:28:35,877 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=zxcvbnmasdfghjkl&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None


Status Code: 202
Headers: {'Server': 'nginx', 'Date': 'Sun, 20 Sep 2026 14:28:35 GMT', 'Content-Type': 'application/x-javascript', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Allow-Origin': '*', 'Server-Timing': 'total;dur=34;desc="Backend Total [n]"', 'X-Robots-Tag': 'noindex', 'Strict-Transport-Security': 'max-age=31536000', 'Permissions-Policy': 'interest-cohort=()', 'Content-Security-Policy': "default-src 'none' ; connect-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; manifest-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzczad.onion/ https://spreadprivacy.com ; media-src  https://duck.ai https://*.duck.ai https://duckduckgo.com https://*.duckduckgo.com https://duckduckgogg42xjoc72x3sjasowoarfbgcmvfimaftt6twagswzcz

## 4. Test the Actual Tool Function

In [7]:
# Copy the exact implementation here for testing
from typing import Optional, Dict, Union

class ToolResponse:
    """Mock ToolResponse for testing"""
    def __init__(self, user_response="", llm_response=""):
        self.user_response = user_response
        self.llm_response = llm_response
        self.logs = []
    
    def add_log(self, message: str, level: str = "INFO"):
        self.logs.append(f"[{level}] {message}")
    
    def __repr__(self):
        return f"ToolResponse(user={len(self.user_response)} chars, logs={len(self.logs)} entries)"

def duckduckgo_search(query: str, max_results: int = 5) -> Union[str, ToolResponse]:
    """Perform a web search using DuckDuckGo.

    Args:
        query: The search query.
        max_results: Maximum number of search results to return.

    Returns:
        A ToolResponse object containing the search results or an error message.
    """
    # Initialize response object for logging
    response_obj = ToolResponse(
        user_response="",
        llm_response=""
    )
    
    try:
        response_obj.add_log(f"Search initiated with query: '{query}', max_results: {max_results}")
        
        # DuckDuckGo API endpoint with proper parameters
        url = "https://api.duckduckgo.com/"
        params = {
            "q": query,
            "format": "json",
            "no_html": 1,
            "skip_disambig": 1,
        }
        response_obj.add_log(f"API endpoint: {url} with params: {params}")
        
        response = requests.get(url, params=params, timeout=10)
        response_obj.add_log(f"HTTP response status: {response.status_code}")
        
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()
        
        response_obj.add_log(f"API response parsed successfully")
        response_obj.add_log(f"Response keys available: {list(data.keys())}")

        results = []
        
        # Extract Abstract if available
        if data.get("AbstractText"):
            abstract_text = data['AbstractText']
            response_obj.add_log(f"AbstractText found: {abstract_text[:100]}..." if len(abstract_text) > 100 else f"AbstractText found: {abstract_text}")
            results.append(f"Abstract: {abstract_text}")
        else:
            response_obj.add_log("No AbstractText found in response")
        
        # Extract Related Topics
        related_topics = data.get("RelatedTopics", [])
        response_obj.add_log(f"RelatedTopics count: {len(related_topics)}")
        
        if related_topics:
            extracted_count = 0
            for idx, topic in enumerate(related_topics[:max_results]):
                response_obj.add_log(f"Processing RelatedTopic[{idx}], keys: {list(topic.keys())}")
                
                if "Text" in topic:
                    result_text = f"- {topic['Text']}"
                    results.append(result_text)
                    extracted_count += 1
                    response_obj.add_log(f"Extracted result from Text field")
                else:
                    response_obj.add_log(f"RelatedTopic[{idx}] missing Text field", level="WARNING")
            
            response_obj.add_log(f"Successfully extracted {extracted_count} results from {len(related_topics)} topics")
        else:
            response_obj.add_log("No RelatedTopics found in response", level="WARNING")

        if not results:
            response_obj.add_log(f"No valid results extracted from API response", level="WARNING")
            response_obj.user_response = f"No relevant results found for '{query}' on DuckDuckGo."
            response_obj.llm_response = f"DuckDuckGo Search Results for '{query}': No results."
            return response_obj
        
        response_obj.add_log(f"Total results compiled: {len(results)}")
        formatted_results = "\\n".join(results)
        response_obj.user_response = f"DuckDuckGo Search Results for '{query}':\\n{formatted_results}"
        response_obj.llm_response = f"DuckDuckGo Search Results for '{query}':\\n{formatted_results}"
        response_obj.add_log(f"Search completed successfully with {len(results)} results")
        
        return response_obj

    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP Error fetching DuckDuckGo search results: {e}")
        response_obj.add_log(f"HTTP Error: {e}", level="ERROR")
        response_obj.add_log(f"Response status: {e.response.status_code}, reason: {e.response.reason}", level="ERROR")
        response_obj.user_response = f"HTTP Error performing DuckDuckGo search: {e}"
        response_obj.llm_response = f"DuckDuckGo Search HTTP Error: {e}"
        return response_obj
    
    except requests.exceptions.Timeout as e:
        logger.error(f"Timeout fetching DuckDuckGo search results: {e}")
        response_obj.add_log(f"Request timeout (10s exceeded): {e}", level="ERROR")
        response_obj.user_response = f"Timeout performing DuckDuckGo search: {e}"
        response_obj.llm_response = f"DuckDuckGo Search Timeout: {e}"
        return response_obj
    
    except requests.exceptions.RequestException as e:
        logger.error(f"Network error fetching DuckDuckGo search results: {e}")
        response_obj.add_log(f"Network error: {e}", level="ERROR")
        response_obj.user_response = f"Network error performing DuckDuckGo search: {e}"
        response_obj.llm_response = f"DuckDuckGo Search Error: {e}"
        return response_obj
    
    except ValueError as e:
        logger.error(f"JSON parsing error during DuckDuckGo search: {e}")
        response_obj.add_log(f"JSON parsing failed: {e}", level="ERROR")
        response_obj.user_response = f"Invalid response format from DuckDuckGo: {e}"
        response_obj.llm_response = f"DuckDuckGo Search Parse Error: {e}"
        return response_obj
    
    except Exception as e:
        logger.error(f"Unexpected error during DuckDuckGo search: {e}")
        response_obj.add_log(f"Unexpected error: {type(e).__name__}: {e}", level="ERROR")
        response_obj.user_response = f"An unexpected error occurred during DuckDuckGo search: {e}"
        response_obj.llm_response = f"DuckDuckGo Search Unexpected Error: {e}"
        return response_obj

print("✅ Tool function loaded successfully")

✅ Tool function loaded successfully


## 5. Run Tool Function Tests

In [8]:
def print_tool_response(response_obj, title=""):
    """Pretty print tool response with logs"""
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}\n")
    
    print("📋 LOGS:")
    for log in response_obj.logs:
        print(f"  {log}")
    
    print(f"\n👤 USER RESPONSE:")
    print(f"{response_obj.user_response}")
    
    print(f"\n🤖 LLM RESPONSE:")
    print(f"{response_obj.llm_response}")

# Test 1: Normal query
result1 = duckduckgo_search("machine learning basics", max_results=3)
print_tool_response(result1, "TEST 1: Normal Query - 'machine learning basics'")

2026-09-20 14:29:56,361 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443
2026-09-20 14:29:56,553 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=machine+learning+basics&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None



  TEST 1: Normal Query - 'machine learning basics'

📋 LOGS:
  [INFO] Search initiated with query: 'machine learning basics', max_results: 3
  [INFO] API endpoint: https://api.duckduckgo.com/ with params: {'q': 'machine learning basics', 'format': 'json', 'no_html': 1, 'skip_disambig': 1}
  [INFO] HTTP response status: 202
  [INFO] API response parsed successfully
  [INFO] Response keys available: ['Abstract', 'AbstractSource', 'AbstractText', 'AbstractURL', 'Answer', 'AnswerType', 'Definition', 'DefinitionSource', 'DefinitionURL', 'Entity', 'Heading', 'Image', 'ImageHeight', 'ImageIsLogo', 'ImageWidth', 'Infobox', 'Redirect', 'RelatedTopics', 'Results', 'Type', 'meta']
  [INFO] No AbstractText found in response
  [INFO] RelatedTopics count: 0
  [WARNING] No RelatedTopics found in response
  [WARNING] No valid results extracted from API response

👤 USER RESPONSE:
No relevant results found for 'machine learning basics' on DuckDuckGo.

🤖 LLM RESPONSE:
DuckDuckGo Search Results for 'machi

In [9]:
# Test 2: Query with max_results parameter
result2 = duckduckgo_search("artificial intelligence", max_results=5)
print_tool_response(result2, "TEST 2: Different max_results - 'artificial intelligence'")

2026-09-20 14:30:24,021 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.duckduckgo.com:443
2026-09-20 14:30:24,177 - urllib3.connectionpool - DEBUG - https://api.duckduckgo.com:443 "GET /?q=artificial+intelligence&format=json&no_html=1&skip_disambig=1 HTTP/1.1" 202 None



  TEST 2: Different max_results - 'artificial intelligence'

📋 LOGS:
  [INFO] Search initiated with query: 'artificial intelligence', max_results: 5
  [INFO] API endpoint: https://api.duckduckgo.com/ with params: {'q': 'artificial intelligence', 'format': 'json', 'no_html': 1, 'skip_disambig': 1}
  [INFO] HTTP response status: 202
  [INFO] API response parsed successfully
  [INFO] Response keys available: ['Abstract', 'AbstractSource', 'AbstractText', 'AbstractURL', 'Answer', 'AnswerType', 'Definition', 'DefinitionSource', 'DefinitionURL', 'Entity', 'Heading', 'Image', 'ImageHeight', 'ImageIsLogo', 'ImageWidth', 'Infobox', 'Redirect', 'RelatedTopics', 'Results', 'Type', 'meta']
  [INFO] AbstractText found: Artificial intelligence is the capability of computational systems to perform tasks typically associ...
  [INFO] RelatedTopics count: 9
  [INFO] Processing RelatedTopic[0], keys: ['FirstURL', 'Icon', 'Result', 'Text']
  [INFO] Extracted result from Text field
  [INFO] Processing Rel

In [ ]:
# Test 3: Nonsense query (edge case)
result3 = duckduckgo_search("xyzabc123notarealquery999", max_results=5)
print_tool_response(result3, "TEST 3: Edge Case - Nonsense Query")

In [ ]:
# Test 4: Very common query
result4 = duckduckgo_search("what is python", max_results=3)
print_tool_response(result4, "TEST 4: Common Query - 'what is python'")

In [ ]:
# Test 5: Question format
result5 = duckduckgo_search("how to learn programming", max_results=2)
print_tool_response(result5, "TEST 5: Question Format - 'how to learn programming'")

## 6. Debugging & Analysis

In [ ]:
# Summary of all tests
print("\n" + "="*60)
print("  TEST SUMMARY")
print("="*60 + "\n")

test_results = [
    ("Machine Learning", result1),
    ("AI (max_results=5)", result2),
    ("Nonsense Query", result3),
    ("What is Python", result4),
    ("How to Learn", result5),
]

for test_name, result in test_results:
    has_response = bool(result.user_response)
    num_logs = len(result.logs)
    error_logs = sum(1 for log in result.logs if "ERROR" in log)
    warning_logs = sum(1 for log in result.logs if "WARNING" in log)
    
    status = "✅" if has_response and error_logs == 0 else "⚠️" if warning_logs > 0 else "❌"
    
    print(f"{status} {test_name:20} | Response: {'Yes' if has_response else 'No':3} | Logs: {num_logs:2} | Errors: {error_logs} | Warnings: {warning_logs}")

## 7. Detailed Debug Inspection

In [ ]:
# Let's do a detailed inspection of one API response to identify any parsing issues
print("\n" + "="*60)
print("  DETAILED API RESPONSE INSPECTION")
print("="*60 + "\n")

url = "https://api.duckduckgo.com/"
params = {
    "q": "python",
    "format": "json",
    "no_html": 1,
    "skip_disambig": 1,
}

response = requests.get(url, params=params, timeout=10)
data = response.json()

print("\n📌 FULL RESPONSE STRUCTURE:")
print(json.dumps(data, indent=2)[:2000] + "\n...\n")

print("\n🔍 RELATED TOPICS DEEP DIVE:")
for idx, topic in enumerate(data.get('RelatedTopics', [])[:2]):
    print(f"\nTopic[{idx}]:")
    print(json.dumps(topic, indent=2))

## 8. Interactive Testing

Use this cell to test any query you want:

In [ ]:
# Interactive test - modify the query below and run
custom_query = "python programming language"  # <-- MODIFY THIS
custom_max_results = 5  # <-- OR MODIFY THIS

result = duckduckgo_search(custom_query, max_results=custom_max_results)
print_tool_response(result, f"CUSTOM TEST: '{custom_query}'")

## 9. Known Issues & Notes

Use this cell to document any issues found during testing:

In [ ]:
"""
ISSUES FOUND:
1. 
2. 
3. 

OBSERVATIONS:
1. API response varies significantly based on query type
2. RelatedTopics field is consistently available
3. AbstractText is not always present (depends on query)

TODO:
1. 
2. 
3. 
"""

print("Ready for debugging notes - update the cell above as needed!")